# MLFlow + GitLab model upload/download

This notebook shows how to interact with the MLFlow-backed GitLab model registry.

Required environment variables:

- `DIGEI_MLFLOW_TRACKING_URI`
- `GL_DIGEI_MLFLOW_TOKEN`

Python package requirements:

- jupyter
- mlflow
- torch


In [ ]:
import os
from mlflow import MlflowClient
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = os.environ["DIGEI_MLFLOW_TRACKING_URI"]
os.environ["MLFLOW_TRACKING_TOKEN"] = os.environ["GL_DIGEI_MLFLOW_TOKEN"]
client = MlflowClient()

experiment = "test_experiment"
mlflow.set_experiment(experiment)

In [ ]:
import torch
from torch import nn
import mlflow.pytorch as mlpt

model = nn.Linear(10, 1)

### Create model

Using MLFlow flavor


In [ ]:
model_name = "another_new_model"
artifact_path = ""  # artifact path must be empty for correct model loading when using mlflow.pytorch


with mlflow.start_run(run_name="test_run") as run:
    info = mlpt.log_model(model, artifact_path, registered_model_name=model_name)

mlpt.load_model(f"models:/{model_name}/latest")

Registered model 'another_new_model' already exists. Creating a new version of this model...
2025/08/27 17:35:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: another_new_model, version 104
Created version '104' of model 'another_new_model'.
2025/08/27 17:35:17 INFO mlflow.tracking._tracking_service.client: 🏃 View run test_run at: https://gitlabdigei.aizoon.it/api/v4/projects/276/ml/mlflow/#/experiments/15/runs/7487c29b-7ebd-475b-b6d6-a287fabdbb5c.
2025/08/27 17:35:17 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://gitlabdigei.aizoon.it/api/v4/projects/276/ml/mlflow/#/experiments/15.


Using MLFlow artifacts and register model


In [ ]:
import mlflow.artifacts

save_location = "test_model.pt"
torch.save(model, save_location)

artifact_path = "model"  # in this case, the artifact path can be empty but we can't use mlflow.pytorch
with mlflow.start_run(run_name="test_run") as run:
    mlflow.log_artifact(save_location, artifact_path)
    mlflow.register_model(
        f"runs:/{run.info.run_id}/{save_location}",
        model_name,
    )

out = mlflow.artifacts.download_artifacts(f"models:/{model_name}/latest")
torch.load(f"{out}/{artifact_path}/{save_location}")

Registered model 'another_new_model' already exists. Creating a new version of this model...
2025/08/28 09:36:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: another_new_model, version 110
Created version '110' of model 'another_new_model'.
2025/08/28 09:36:57 INFO mlflow.tracking._tracking_service.client: 🏃 View run test_run at: https://gitlabdigei.aizoon.it/api/v4/projects/276/ml/mlflow/#/experiments/15/runs/41678c66-c4d8-4d66-a176-13ce238832fd.
2025/08/28 09:36:57 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://gitlabdigei.aizoon.it/api/v4/projects/276/ml/mlflow/#/experiments/15.
c:\Users\giovanni.graziano\Anaconda3\envs\pv-training\lib\site-packages\mlflow\store\artifact\utils\models.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about 

Linear(in_features=10, out_features=1, bias=True)

Using MLFlow client API I could not get it to upload artifacts, this results in a 404.

The only way I found was to log the model or artifacts directly to the run associated with the registered model


In [ ]:
with mlflow.start_run(run_name="test_run") as run:
    mlpt.log_model(model, artifact_path)  # This will not register a new model

model_version = "2.0.5"  # this might result in an error if the version exists. I do not have delete privileges

mlflow.set_experiment(experiment)

tags = {"gitlab.version": model_version}
artifact_path = ""
mv_info = client.create_model_version(
    model_name, f"runs:/{run.info.run_id}/{artifact_path}", tags=tags
)

mlflow.set_experiment(f"[model]{model_name}")
with mlflow.start_run(run_id=mv_info.run_id) as run:
    mlpt.log_model(model, artifact_path)  # This will not register a new model
mlpt.load_model(f"models:/{model_name}/{model_version}")

2025/08/28 10:23:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/08/28 10:23:18 WARNING mlflow.models.model: Logging model metadata to the tracking server has failed. The model artifacts have been logged successfully under mlflow-artifacts:/candidate:60. Set logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)` to see the full traceback.
2025/08/28 10:23:18 INFO mlflow.tracking._tracking_service.client: 🏃 View run test_run at: https://gitlabdigei.aizoon.it/api/v4/projects/276/ml/mlflow/#/experiments/7/runs/bf6c2a53-8b07-40fa-896f-0c24a0b2c673.
2025/08/28 10:23:18 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://gitlabdigei.aizoon.it/api/v4/projects/276/ml/mlflow/#/experiments/7.
2025/08/28 10:23:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finis

Linear(in_features=10, out_features=1, bias=True)